# A4 — Self-Organizing

**Recursive delegation: the manager can promote workers to submanagers with a reserved budget slice.**

**Fictional task**: prepare a multi-angle launch plan for a fictional weather-monitoring cubesat (regulatory / technical / logistics).

In [ ]:
# --- Load API key from the canonical env file (see memory `reference_api_keys`) ---
import os
from pathlib import Path

env_file = Path('/home/shumway/projects/meta-agents/.env')
if env_file.exists() and not os.environ.get('OPENROUTER_API_KEY'):
    for raw in env_file.read_text().splitlines():
        s = raw.strip()
        if s.startswith('OPENROUTER_API_KEY='):
            os.environ['OPENROUTER_API_KEY'] = s.split('=', 1)[1].strip().strip('"').strip("'")
            break

assert os.environ.get('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY missing'
# Default worker model for agents that do not declare their own (DAG engine consults LLM_MODEL).
os.environ.setdefault('LLM_MODEL', 'deepseek/deepseek-chat-v3.1')
print('OpenRouter key loaded. Default model:', os.environ['LLM_MODEL'])

## Load + A4 compliance

In [ ]:
from pathlib import Path
from awp.parser import parse_manifest, parse_agent
from awp.validator import check_compliance, AutonomyLevel

WORKFLOW_DIR = Path('/home/shumway/projects/agent-workflow-protocol/examples/workflows/09-recursive-delegation')
manifest = parse_manifest(WORKFLOW_DIR / 'workflow.awp.yaml')
agents = {}
for ad in (WORKFLOW_DIR / 'agents').iterdir():
    a = ad / 'agent.awp.yaml'
    if a.exists():
        agents[ad.name] = parse_agent(a)

result = check_compliance(manifest, agents, target_level=AutonomyLevel.A4_SELF_ORGANIZING)
assert result.level >= AutonomyLevel.A4_SELF_ORGANIZING, f'Not A4: {result.errors}'
budget = manifest.orchestration.delegation_loop.budget
print(f'A4 compliant. max_depth={budget.max_depth} allows submanager spawning.')

## Run the recursive delegation

In [ ]:
import json
import logging
logging.basicConfig(level=logging.WARNING)

from awp.runtime import WorkflowRunner

TASK = (
    'Prepare a first-draft launch plan for a fictional weather-monitoring cubesat. '
    'Cover three angles — regulatory, technical, and logistics — and return a short consolidated summary with one key action item per angle.'
)
runner = WorkflowRunner(
    WORKFLOW_DIR,
    manager_model='openai/gpt-5-mini',
    worker_model='deepseek/deepseek-chat-v3.1',
)
result = runner.run(TASK)

print(json.dumps({k: v for k, v in result.items() if not k.startswith('_')}, indent=2, default=str)[:3500])

## Assertions (E2E rubric)

In [ ]:
assert isinstance(result, dict) and result, 'empty result'
has_content = any(
    (isinstance(v, (str, dict, list)) and bool(v)) for k, v in result.items() if not k.startswith('_')
)
assert has_content, f'no content in result. Keys: {list(result)}'
data_dir = WORKFLOW_DIR / 'data'
if data_dir.exists():
    for kind in ('traces', 'audit', 'metrics'):
        p = data_dir / kind
        if p.exists():
            files = list(p.iterdir())
            print(f'observability/{kind}: {len(files)} file(s)')
print('A4 OK — recursive delegation closed.')